# FHIR on RAG

This notebook is loading FHIR resources into a vector store and then using that to help prompt an LLM to answer questions about the data. To do that, it first flattens the FHIR resources into text files. It then uses [LlamaIndex](https://www.llamaindex.ai/) to load the text files into an in-memory vector store. Then it calls out to a LLama 2 running locally using [Ollama](https://ollama.ai/) using different [strategies](https://docs.llamaindex.ai/en/stable/module_guides/querying/response_synthesizers/root.html) for combining the FHIR with the question into the prompt.

In [49]:
# Some constants to use throughout 

in_file_glob = './working/raw_fhir/*.json'
flat_file_path = './working/flat'
vector_store_file_path = './working/vector_store'

## Flatten FHIR

This is going to read in any JSON files in the `in_file_glob`. It assumes that each file is a FHIR Bundle. It will first pull out the Patient resource and extract some key information, like name, from it to include in the text files it will create per resource. This helps the RAG know which patient a resource goes with. It then flattens each resource in the bundle. 

Flattening it means that it creates a path of all the attribute names from the root of the resource to each value. In the process it splits any camel case words into multiple words. Finally, it writes this out to a text file in the structure of:
``` [path name] is [value]. ```
This creates a semi-english version of the resource that can be turned into a vector by the embedding. 

**To use this project,** you will need to create the working and raw_fhir directories and populate raw_fhir with FHIR Bundles. I used [Synthea](https://synthea.mitre.org/) to generate synthetic data in my testing.

In [50]:
import glob
import os
import json
import re

camel_pattern1 = re.compile(r'(.)([A-Z][a-z]+)')
camel_pattern2 = re.compile(r'([a-z0-9])([A-Z])')

def exclude_references(flat_entry):
    return {k: v for k, v in flat_entry.items() if 'reference' not in k.lower()}
def split_camel(text):
    new_text = camel_pattern1.sub(r'\1 \2', text)
    new_text = camel_pattern2.sub(r'\1 \2', new_text)
    return new_text


def handle_special_attributes(attrib_name, value):
    if attrib_name == 'resource Type':
        return split_camel(value)
    return value


def flatten_fhir(nested_json):
    out = {}

    def flatten(json_to_flatten, name=''):
        if type(json_to_flatten) is dict:
            json_to_flatten = exclude_references(json_to_flatten)
            for sub_attribute in json_to_flatten:
                flatten(json_to_flatten[sub_attribute], name + split_camel(sub_attribute) + ' ')
        elif type(json_to_flatten) is list:
            for i, sub_json in enumerate(json_to_flatten):
                flatten(sub_json, name + str(i) + ' ')
        else:
            attrib_name = name[:-1]
            out[attrib_name] = handle_special_attributes(attrib_name, json_to_flatten)

    flatten(nested_json)
    return out


def filter_for_patient(entry):
    return entry['resource']['resourceType'] == "Patient"


def find_patient(bundle):
    patients = list(filter(filter_for_patient, bundle['entry']))
    if len(patients) < 1:
        raise Exception('No Patient found in bundle!')
    else:
        patient = patients[0]['resource']

        patient_id = patient['id']
        first_name = patient['name'][0]['given'][0]
        last_name = patient['name'][0]['family']

        return {'PatientFirstName': first_name, 'PatientLastName': last_name, 'PatientID': patient_id}


def flat_to_string(flat_entry):
    output = ''

    for attrib in flat_entry:
        output += f'{attrib} is {flat_entry[attrib]}. '

    return output


def flatten_bundle(bundle_file_name):
    file_name = bundle_file_name[bundle_file_name.rindex('/') + 1:bundle_file_name.rindex('.')]
    with open(bundle_file_name) as raw:
        bundle = json.load(raw)
        patient = find_patient(bundle)
        flat_patient = flatten_fhir(patient)
        for i, entry in enumerate(bundle['entry']):
            flat_entry = flatten_fhir(entry['resource'])
            with open(f'{flat_file_path}/{file_name}_{i}.txt', 'w') as out_file:
                out_file.write(f'{flat_to_string(flat_patient)}\n{flat_to_string(flat_entry)}')


if not os.path.exists(flat_file_path):
    os.mkdir(flat_file_path)

for file in glob.glob(in_file_glob):
    flatten_bundle(file)

## Setup the Gen AI with RAG

This section will use LlamaIndex to construct the vector store and tie to the LLM. 

In [51]:
!pip install llama-index
!pip install transformers
!pip install llama-index-embeddings-huggingface
!pip install llama-index-llms-ollama

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


I tried a couple of different models for doing the embedding, i.e. turning the flattened FHIR text into vectors. I would like to experement with others, but haven't had time. In the end, `BAAI/bge-large-en-v1.5` was too big for me to run on my local, so I did most of my testing with `BAAI/bge-small-en-v1.5`.

In [52]:

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# loads BAAI/bge-small-en
# embed_model = HuggingFaceEmbedding()

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-large-en-v1.5")

# embed_model = HuggingFaceEmbedding(model_name="medicalai/ClinicalBERT")

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'text']
2 prompts are loaded, with the keys: ['query', 'text']
2 prompts are loaded, with the keys: ['query', 'text']
2 prompts are loaded, with the keys: ['query', 'text']
2 prompts are loaded, with the keys: ['query', 'text']
2 prompts are loaded, with the keys: ['query', 'text']


In [53]:
# This code is to play with embedding if desired. It is not needed and can remain commented out.

# embeddings = embed_model.get_text_embedding("Hello World!")
# print(len(embeddings))
# print(embeddings[:5])

In [77]:
from llama_index.llms.ollama import Ollama

# LLama 2 is running locally, using Ollama.
llm = Ollama(model="llama3.2:3b-instruct-q8_0", request_timeout=300)

In [55]:
# This is a test prompt, just to prove that Ollama is working. It can remain commented out.

# resp = llm.complete("Who is Paul Graham?")
# print(resp)

In [78]:
# --- OLD CODE (Causes Error) ---
# from llama_index import ServiceContext, VectorStoreIndex, SimpleDirectoryReader, SummaryIndex
# from llama_index import set_global_service_context
# service_context = ServiceContext.from_defaults(llm=llm, embed_model=embed_model)
# set_global_service_context(service_context) # Also deprecated

# --- NEW CODE (Corrected) ---
from llama_index.core import (
    Settings,
    VectorStoreIndex,
    SimpleDirectoryReader,
    SummaryIndex # Assuming SummaryIndex is also in core, adjust if needed
)
# Assuming 'llm' and 'embed_model' are already defined LlamaIndex LLM and Embedding objects
# Configure global settings instead of using ServiceContext
Settings.llm = llm
Settings.embed_model = embed_model

# Now you can create indices or query engines, and they will use the models from Settings
# Example:
# documents = SimpleDirectoryReader(...).load_data()
# index = VectorStoreIndex.from_documents(documents) # Will use Settings.embed_model
# query_engine = index.as_query_engine() # Will use Settings.llm and Settings.embed_model


In [57]:
# This code loads the flat FHIR text files. 

documents = SimpleDirectoryReader(flat_file_path).load_data()
print(len(documents))

1124


In [58]:
# Load those flat FHIR text files into the vector store.

vector_index = VectorStoreIndex.from_documents(documents, show_progress=True)


# if not os.path.exists(vector_store_file_path):
#     os.mkdir(vector_store_file_path)
# vector_index.vector_store.persist(f'{vector_store_file_path}/FHIR_RAG.vs')

Generating embeddings: 100%|██████████| 1790/1790 [08:06<00:00,  3.68it/s]


In [59]:
# I tried to play with summary indexes, but it took too long to get a response on my machine. 

# summary_index = SummaryIndex.from_documents(documents)

In [60]:
from llama_index.core.response.notebook_utils import display_response
import logging
import sys
from IPython.core.display import Markdown

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

## Actually do RAG

This is the code block that actually asks the questions of the LLM. 

In my tests, I used synthetic FHIR generated by [Synthea](https://github.com/synthetichealth/synthea/wiki/Basic-Setup-and-Running). I had Synthea generate two patients and so I asked questions about each patient. If you are looking to replicate this work, you will need to change the patient names to match what ever data you have available. 

In [79]:
from requests.exceptions import ReadTimeout

def display_source_text(response):
    for ind, source_node in enumerate(response.source_nodes):
        display(Markdown("---"))
        display(Markdown(f"**`Source Node {ind + 1}/{len(response.source_nodes)}`**"))
        text_md = (
            f'**File:** {source_node.node.metadata["file_name"]}<br>'
            f'**Text:** {source_node.node.get_content().strip()}'
        )
        display(Markdown(text_md))


def ask_question(index, response_mode, question, show_sources=False):
    query_engine = index.as_query_engine(response_mode=response_mode, similarity_top_k=5)
    try:
        response = query_engine.query(question)  # Removed unsupported request_timeout argument
    except ReadTimeout:
        display(Markdown(f"### Error: The query timed out while processing the question: {question}"))
        return
    display(Markdown(f'### Answer for {response_mode}'))
    if show_sources:
        display_source_text(response)
    else:
        display_response(response, show_source=False, show_metadata=False, show_source_metadata=False)


def ask_question_all_modes(person, question):
    display(Markdown(f'# Asking about {person}\n<br>**Question:** {question}'))
    ask_question(vector_index, 'no_text', question, show_sources=True)
    #ask_question(vector_index, 'simple_summarize', question)
    ask_question(vector_index, 'compact', question)
    #ask_question(vector_index, 'refine', question)
    #ask_question(vector_index, 'tree_summarize', question)
    #ask_question(vector_index, 'accumulate', question)
    #ask_question(vector_index, 'compact_accumulate', question)


ask_question_all_modes('Alex',
                      'What can you tell me about Alex454  well child visit?')
#ask_question_all_modes('Ammie', 'What can you tell me about Ammie189 history?')

# Asking about Alex
<br>**Question:** What can you tell me about Alex454  well child visit?

### Answer for no_text

---

**`Source Node 1/5`**

**File:** Alex454_White193_2f12e0bd-98b8-8ba2-643a-6aca9799cbb2_7.txt<br>**Text:** Patient First Name is Alex454. Patient Last Name is White193. Patient ID is 2f12e0bd-98b8-8ba2-643a-6aca9799cbb2. 
resource Type is Encounter. id is 373bdaee-d8e6-5758-8f0e-cf4846037445. meta profile 0 is http://hl7.org/fhir/us/core/StructureDefinition/us-core-encounter. identifier 0 use is official. identifier 0 system is https://github.com/synthetichealth/synthea. identifier 0 value is 373bdaee-d8e6-5758-8f0e-cf4846037445. status is finished. class system is http://terminology.hl7.org/CodeSystem/v3-ActCode. class code is AMB. type 0 coding 0 system is http://snomed.info/sct. type 0 coding 0 code is 410620009. type 0 coding 0 display is Well child visit (procedure). type 0 text is Well child visit (procedure). subject display is Mr. Alex454 Jake169 White193. participant 0 type 0 coding 0 system is http://terminology.hl7.org/CodeSystem/v3-ParticipationType. participant 0 type 0 coding 0 code is PPRF. participant 0 type 0 coding 0 display is primary performer. participant 0 type 0 text is primary performer. participant 0 period start is 1985-11-29T23:31:24+00:00. participant 0 period end is 1985-11-29T23:52:35+00:00. participant 0 individual display is Dr. Norah104 Jenkins714. period start is 1985-11-29T23:31:24+00:00. period end is 1985-11-29T23:52:35+00:00. location 0 location display is CHARLES RIVER COMMUNITY HEALTH, INC. service Provider display is CHARLES RIVER COMMUNITY HEALTH, INC.

...

---

**`Source Node 5/5`**

**File:** Alex454_White193_2f12e0bd-98b8-8ba2-643a-6aca9799cbb2_36.txt<br>**Text:** Patient First Name is Alex454. Patient Last Name is White193. Patient ID is 2f12e0bd-98b8-8ba2-643a-6aca9799cbb2. 
resource Type is Encounter. id is 49db4644-1b7f-5205-ae48-e6d4858948e7. meta profile 0 is http://hl7.org/fhir/us/core/StructureDefinition/us-core-encounter. identifier 0 use is official. identifier 0 system is https://github.com/synthetichealth/synthea. identifier 0 value is 49db4644-1b7f-5205-ae48-e6d4858948e7. status is finished. class system is http://terminology.hl7.org/CodeSystem/v3-ActCode. class code is AMB. type 0 coding 0 system is http://snomed.info/sct. type 0 coding 0 code is 410620009. type 0 coding 0 display is Well child visit (procedure). type 0 text is Well child visit (procedure). subject display is Mr. Alex454 Jake169 White193. participant 0 type 0 coding 0 system is http://terminology.hl7.org/CodeSystem/v3-ParticipationType. participant 0 type 0 coding 0 code is PPRF. participant 0 type 0 coding 0 display is primary performer. participant 0 type 0 text is primary performer. participant 0 period start is 1987-12-11T23:31:24+00:00. participant 0 period end is 1987-12-11T23:46:24+00:00. participant 0 individual display is Dr. Norah104 Jenkins714. period start is 1987-12-11T23:31:24+00:00. period end is 1987-12-11T23:46:24+00:00. location 0 location display is CHARLES RIVER COMMUNITY HEALTH, INC. service Provider display is CHARLES RIVER COMMUNITY HEALTH, INC.

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"


ResponseError: model requires more system memory (5.8 GiB) than is available (3.9 GiB) (status code: 500)

In [80]:
from llama_index.core import PromptTemplate
# ====== Customise prompt template ======
qa_prompt_tmpl_str = (
"Context information is below.\n"
"---------------------\n"
"{context_str}\n"
"---------------------\n"
"Given the context information above I want you to think step by step to answer the query in a crisp manner, incase case you don't know the answer say 'I don't know!'.\n"
"Query: {query_str}\n"
"Answer: "
)
qa_prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)
query_engine = vector_index.as_query_engine()

query_engine.update_prompts(
    {"response_synthesizer:text_qa_template": qa_prompt_tmpl}
)

# Generate the response
response = query_engine.query("What exactly is DSPy?",)

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"


ResponseError: model requires more system memory (5.8 GiB) than is available (3.9 GiB) (status code: 500)